# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
%pip install "langchain==0.2.17" "langchain-core==0.2.43" "langchain-community==0.2.19" "langchain-openai==0.1.25" "langchain-elasticsearch==0.2.2" "jq==1.12.0" "tiktoken>=0.7,<1" "python-dotenv>=1,<2"


The dependencies are pinned to the LangChain 0.2 API used in this lab. Restart the notebook kernel after installation if it previously loaded a different version. Run the cells in order; credentials are read from environment variables or secure prompts. The indexing and question-answering cells call external services.


In [1]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

# Environment variables are optional; otherwise enter credentials securely.
ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID") or getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY") or getpass("Elastic API key: ")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")
INDEX_NAME = "workplace-multi-query-rag"

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [3]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"
with urlopen(url, timeout=30) as response:
    data = json.load(response)

with open("temp.json", "w", encoding="utf-8") as json_file:
    json.dump(data, json_file, ensure_ascii=False)

print(f"Loaded {len(data)} source documents.")

Loaded 15 source documents.


### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [4]:
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    for key in ("name", "summary", "url", "category", "updated_at"):
        metadata[key] = record.get(key, "")
    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400
)
docs = loader.load_and_split(text_splitter=text_splitter)
print(f"Split {len(data)} documents into {len(docs)} passages.")

Split 15 documents into 15 passages.


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [5]:
import hashlib

# Stable IDs keep rerunning this cell from duplicating the same passages.
passage_ids = [
    hashlib.sha256(
        f"{doc.metadata['source']}:{doc.metadata['seq_num']}:{i}:{doc.page_content}".encode("utf-8")
    ).hexdigest()
    for i, doc in enumerate(docs)
]
vectorstore.add_documents(documents=docs, ids=passage_ids)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm,
    include_original=True,
)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [6]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
from langchain_core.output_parsers import StrOutputParser

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Explain your answer clearly and cite source names. When policies conflict, use the most recent dated update in the context. Never invent missing policy values. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
UPDATED: {updated_at}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the NASA sales team do and what are their responsibilities?  ', 'Can you provide information about the structure and functions of the NASA sales team?  ', 'What roles and activities are associated with the sales team at NASA?']
---- Answer ----
The NASA sales team refers to the North America South America region within the company's sales organization. This region includes the United States, Canada, Mexico, as well as Central and South America. The team is led by two Area Vice-Presidents: Laura Martinez oversees North America, while Gary Johnson is responsible for South America. The sales team in this region is tasked with identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction across these diverse markets (Sales Organization Overview).


<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

## Iteration 1: Hybrid work policy

**Question:** According to the May 2023 update, how many days per week must employees work in the office, and who should they coordinate with?

This tests whether retrieval can find a dated update despite the older remote-work policies in the dataset.

In [7]:
question_1 = (
    "According to the May 2023 update, how many days per week must employees "
    "work in the office, and who should they coordinate with?"
)
answer_1 = chain.invoke(question_1)
print("---- Hybrid work answer ----")
print(answer_1)

---- Hybrid work answer ----
According to the May 2023 update, employees must work from the office three days a week, with two days designated for remote work. Employees should coordinate with their supervisor and the HR department to establish their updated in-office workdays (Wfh Policy Update May 2023).


### Reflection 1

The source document **Wfh Policy Update May 2023** specifies three office days and two remote days per week from May 1, 2023. Employees should coordinate their office days with their supervisor and HR. A grounded answer should cite this update instead of using the older two-day or fully remote policy.

Useful query variations would preserve the May 2023 date and use synonyms such as "hybrid work schedule" and "in-office requirement." Dropping the date could bring back superseded policies. Multi-query retrieval broadens coverage, but does not itself resolve conflicting policies; the answer prompt explicitly prioritizes the latest dated update.

**Execution status:** Live queries have not been run because service credentials are unavailable. These are source-based expectations, not observations of generated queries or model answers. After running, compare the INFO logs and answer above against these criteria.

## Iteration 2: Sales objectives

**Question:** What revenue growth, market-share growth, and customer retention targets are set in the fiscal year 2024 sales strategy?

This tests retrieval of several exact numerical targets from one strategy document.

In [8]:
question_2 = (
    "What revenue growth, market-share growth, and customer retention targets "
    "are set in the fiscal year 2024 sales strategy?"
)
answer_2 = chain.invoke(question_2)
print("---- Sales strategy answer ----")
print(answer_2)

---- Sales strategy answer ----
In the fiscal year 2024 sales strategy, the targets set are as follows:

- **Revenue Growth**: Increase revenue by 20% compared to fiscal year 2023.
- **Market-Share Growth**: Expand market share in key segments by 15%.
- **Customer Retention**: Retain 95% of existing customers and increase customer satisfaction ratings.

These objectives are outlined in the "Fy2024 Company Sales Strategy" document, which was last updated on April 15, 2023.


### Reflection 2

The **Fy2024 Company Sales Strategy** sets targets of 20% revenue growth compared with FY2023, 15% market-share growth in key segments, and 95% customer retention. The answer should retain those distinctions and cite the strategy document.

Useful reformulations would include "FY2024 sales objectives" or "revenue and retention KPIs" while preserving the fiscal year and all three requested measures. Query drift toward sales-team structure would be less useful. Multiple searches may return overlapping passages; MultiQueryRetriever merges duplicate documents, but it cannot guarantee an improvement over a single query.

**Execution status:** Live queries have not been run because service credentials are unavailable. The expected values were checked against the dataset. Actual query usefulness and answer grounding must be assessed from the generated logs and answer when this cell runs.